In [1]:
!cp HiRAG/config.yaml .


In [1]:
import os
import logging
import numpy as np
import yaml
from hirag import HiRAG, QueryParam
from openai import AsyncOpenAI, OpenAI
from dataclasses import dataclass
from hirag.base import BaseKVStorage
from hirag._utils import compute_args_hash

# Load configuration from YAML file
with open('config.yaml', 'r') as file:
    config = yaml.safe_load(file)

# Extract configurations
OPENAI_EMBEDDING_MODEL = config['openai']['embedding_model']
OPENAI_MODEL = config['openai']['model']
OPENAI_API_KEY = config['openai']['api_key']
OPENAI_URL = config['openai']['base_url']
GLM_API_KEY = config['glm']['api_key']
GLM_MODEL = config['glm']['model']
GLM_URL = config['glm']['base_url']

@dataclass
class EmbeddingFunc:
    embedding_dim: int
    max_token_size: int
    func: callable

    async def __call__(self, *args, **kwargs) -> np.ndarray:
        return await self.func(*args, **kwargs)

def wrap_embedding_func_with_attrs(**kwargs):
    """Wrap a function with attributes"""

    def final_decro(func) -> EmbeddingFunc:
        new_func = EmbeddingFunc(**kwargs, func=func)
        return new_func

    return final_decro

@wrap_embedding_func_with_attrs(embedding_dim=config['model_params']['openai_embedding_dim'], max_token_size=config['model_params']['max_token_size'])
async def OPENAI_embedding(texts: list[str]) -> np.ndarray:
    openai_async_client = AsyncOpenAI(base_url=OPENAI_URL, api_key=OPENAI_API_KEY)
    response = await openai_async_client.embeddings.create(
        model=OPENAI_EMBEDDING_MODEL, input=texts, encoding_format="float"
    )
    return np.array([dp.embedding for dp in response.data])

@wrap_embedding_func_with_attrs(embedding_dim=config['model_params']['glm_embedding_dim'], max_token_size=config['model_params']['max_token_size'])
async def GLM_embedding(texts: list[str]) -> np.ndarray:
    model_name = config['glm']['embedding_model']
    client = OpenAI(
        api_key=GLM_API_KEY,
        base_url=GLM_URL
    ) 
    embedding = client.embeddings.create(
        input=texts,
        model=model_name,
    )
    final_embedding = [d.embedding for d in embedding.data]
    return np.array(final_embedding)

async def OPENAI_model_if_cache(
    prompt, system_prompt=None, history_messages=[], **kwargs
) -> str:
    openai_async_client = AsyncOpenAI(
        api_key=OPENAI_API_KEY, base_url=OPENAI_URL
    )
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})

    # Get the cached response if having-------------------
    hashing_kv: BaseKVStorage = kwargs.pop("hashing_kv", None)
    messages.extend(history_messages)
    messages.append({"role": "user", "content": prompt})
    if hashing_kv is not None:
        args_hash = compute_args_hash(OPENAI_MODEL, messages)
        if_cache_return = await hashing_kv.get_by_id(args_hash)
        if if_cache_return is not None:
            return if_cache_return["return"]
    # -----------------------------------------------------

    response = await openai_async_client.chat.completions.create(
        model=OPENAI_MODEL, messages=messages, **kwargs
    )

    # Cache the response if having-------------------
    if hashing_kv is not None:
        await hashing_kv.upsert(
            {args_hash: {"return": response.choices[0].message.content, "model": OPENAI_MODEL}}
        )
    # -----------------------------------------------------
    return response.choices[0].message.content



In [2]:
graph_func = HiRAG(working_dir=config['hirag']['working_dir'],
                      enable_llm_cache=config['hirag']['enable_llm_cache'],
                      embedding_func=OPENAI_embedding,
                      best_model_func=OPENAI_model_if_cache,
                      cheap_model_func=OPENAI_model_if_cache,
                      enable_hierachical_mode=config['hirag']['enable_hierachical_mode'], 
                      embedding_batch_num=config['hirag']['embedding_batch_num'],
                      embedding_func_max_async=config['hirag']['embedding_func_max_async'],
                      enable_naive_rag=config['hirag']['enable_naive_rag'])


2025-03-31 00:03:57,430 - Load KV full_docs with 0 data
2025-03-31 00:03:57,432 - Load KV text_chunks with 0 data
2025-03-31 00:03:57,433 - Load KV community_reports with 0 data
2025-03-31 00:03:57,438 - Loaded graph from tmp/graph_chunk_entity_relation.graphml with 0 nodes, 0 edges
2025-03-31 00:03:57,440 - Load (0, 1536) data
2025-03-31 00:03:57,441 - Init {'embedding_dim': 1536, 'metric': 'cosine', 'storage_file': 'tmp/vdb_entities.json'} 0 data
2025-03-31 00:03:57,442 - Load (0, 1536) data
2025-03-31 00:03:57,443 - Init {'embedding_dim': 1536, 'metric': 'cosine', 'storage_file': 'tmp/vdb_chunks.json'} 0 data


In [3]:
import nest_asyncio
nest_asyncio.apply()

In [4]:
with open("spotify-reviews.csv") as f:
    graph_func.insert(f.read())

2025-03-31 00:04:04,537 - [New Docs] inserting 1 docs
2025-03-31 00:04:05,722 - [New Chunks] inserting 312 chunks
2025-03-31 00:04:05,723 - Insert chunks for naive RAG
2025-03-31 00:04:05,724 - Inserting 312 vectors to chunks
2025-03-31 00:04:08,258 - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-03-31 00:04:08,372 - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-03-31 00:04:08,390 - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-03-31 00:04:08,637 - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-03-31 00:04:08,643 - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-03-31 00:04:08,646 - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-03-31 00:04:09,573 - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-03-31 00:04:09,730 - HTTP Request: POST https://api.openai.com/v1/emb

2025-03-31 00:04:41,792 - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


2025-03-31 00:04:42,416 - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


2025-03-31 00:04:42,589 - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


2025-03-31 00:04:43,540 - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


...

2025-03-31 00:37:23,609 - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-03-31 00:37:23,613 - JSON data successfully extracted.


2025-03-31 00:37:24,032 - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-03-31 00:37:24,046 - JSON data successfully extracted.


2025-03-31 00:37:25,121 - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-03-31 00:37:25,147 - JSON data successfully extracted.


⠴ Processed 225 communities


2025-03-31 00:37:25,831 - Writing graph with 1241 nodes, 2377 edges


In [5]:
print(graph_func.query("what are the top issues related to playlist?", param=QueryParam(mode="hi")))

2025-03-31 00:37:26,921 - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-03-31 00:37:27,210 - Using 20 entites, 24 communities, 173 reasoning path items, 16 text units
2025-03-31 00:37:27,213 - ====== References ======:
Entities (20): ['"PLAYLIST"', '"PLAYLISTS"', '"PLAYLIST SORTING"', '"THE PLAYLIST"', '"THE PLAYLIST ALGORITHM"', '"MICROPLAYLISTING"', '"THE ALBUM/PLAYLIST"', '"A PLAYLIST"', '"PLAYLIST CREATION"', '"MUSIC PLAYLIST"', '"SPOTIFY PLAYLIST"', '"PAYMENT ISSUES"', '"RECOMMENDED PLAYLISTS"', '"DISCOVERY PLAYLIST"', '"EAT MY PLAYLIST FEATURE"', '"QUEUE FUNCTION"', '"SPOTIFY"', '"TURKISH TRACKS"', '"PLAYLIST DUPLICATION"', '"AI PLAYLIST"']

Communities (level, cluster_id) (24): [(0, 'Cluster 12'), (1, 'Cluster 122'), (0, 'Cluster 11'), (2, 'Cluster 210'), (1, 'Cluster 100'), (2, 'Cluster 182'), (0, 'Cluster 45'), (0, 'Cluster 47'), (1, 'Cluster 152'), (1, 'Cluster 101'), (1, 'Cluster 108'), (2, 'Cluster 197'), (1, 'Cluster 146'), (1, 'Cluster 107

"Spotify is a prominent music streaming platform that offers a range of features centered around playlists, which are a critical component of its user experience. Despite their importance, numerous issues and user frustrations have been highlighted regarding Spotify's playlist functionalities.\n\n### Playlist Control and Customization\n\nOne of the primary concerns users have with Spotify's playlists is the lack of control and customization. Users frequently complain about Spotify's tendency to auto-add songs to their playlists, which can disrupt the personalized experience users expect. This automatic inclusion of songs is particularly unsettling for those who desire precise control over their music selection and playlist content. The inability to pin more than a few playlists and unexpected alterations to playlists also present a significant challenge to user satisfaction.\n\n### Shuffle Features and Algorithmic Concerns\n\nAnother prevalent issue is with Spotify's shuffle feature. U

In [6]:
print("Spotify is a prominent music streaming platform that offers a range of features centered around playlists, which are a critical component of its user experience. Despite their importance, numerous issues and user frustrations have been highlighted regarding Spotify's playlist functionalities.\n\n### Playlist Control and Customization\n\nOne of the primary concerns users have with Spotify's playlists is the lack of control and customization. Users frequently complain about Spotify's tendency to auto-add songs to their playlists, which can disrupt the personalized experience users expect. This automatic inclusion of songs is particularly unsettling for those who desire precise control over their music selection and playlist content. The inability to pin more than a few playlists and unexpected alterations to playlists also present a significant challenge to user satisfaction.\n\n### Shuffle Features and Algorithmic Concerns\n\nAnother prevalent issue is with Spotify's shuffle feature. Users have expressed dissatisfaction with the shuffle function not providing a truly random experience. The algorithmic shuffle, which bases its song selection on listening habits, often does not align with the expectations of unpredictability that users desire. This feature is intended to keep the listening experience fresh and varied but has been criticized for prioritizing frequently played tracks, thus failing to meet the needs for diversity in playback.\n\n### Recent Updates and Their Impact\n\nRecent updates to the platform have further compounded user frustrations by affecting how playlists operate. Changes have been made that impact automatic song additions and alter shuffle modes, leading to user dissatisfaction. The implementation of forced shuffle mode for some playlists, particularly in the free-tier version, disrupts the intended order of play for users who meticulously curate their playlists in specific sequences.\n\n### User Feedback and Suggestions\n\nUser feedback underscores a critical need for Spotify to address these issues by enhancing user autonomy over playlist management and improving algorithmic functions. Many turn to forums and reviews to express what changes they hope to see, such as an option to disable unwanted song additions and improvements in recommendation accuracy. The feedback loop facilitated by platforms like the Spotify Forum is invaluable in helping Spotify understand user needs and potential areas for service enhancement.\n\n### Comparison with Competitors\n\nFinally, comparisons are often drawn between Spotify and other music streaming services like Apple Music and YouTube Music, especially concerning playlist management and the seamlessness of shuffle features. These comparisons illustrate that while Spotify's playlists contribute substantially to its appeal, they also indicate where it might lag behind competitors due to user-experience issues. Consistent alignment with user expectations and the ability to maintain satisfaction through improved functionality remain areas of focus for future updates. \n\nIn conclusion, while playlists are undeniably core to Spotify's platform, the challenges surrounding customization, shuffle reliability, and recent service updates must be addressed to enhance user satisfaction and maintain competitive advantages in the music streaming industry.")

Spotify is a prominent music streaming platform that offers a range of features centered around playlists, which are a critical component of its user experience. Despite their importance, numerous issues and user frustrations have been highlighted regarding Spotify's playlist functionalities.

### Playlist Control and Customization

One of the primary concerns users have with Spotify's playlists is the lack of control and customization. Users frequently complain about Spotify's tendency to auto-add songs to their playlists, which can disrupt the personalized experience users expect. This automatic inclusion of songs is particularly unsettling for those who desire precise control over their music selection and playlist content. The inability to pin more than a few playlists and unexpected alterations to playlists also present a significant challenge to user satisfaction.

### Shuffle Features and Algorithmic Concerns

Another prevalent issue is with Spotify's shuffle feature. Users have

In [10]:
print(graph_func.query("issues with advertisements that users are facing and recommendations to improve experience? be factfull and cite. ", param=QueryParam(mode="hi")))

2025-03-31 01:37:23,614 - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-03-31 01:37:23,877 - Using 20 entites, 24 communities, 225 reasoning path items, 16 text units
2025-03-31 01:37:23,880 - ====== References ======:
Entities (20): ['"ADVERTISING"', '"FREQUENT ADVERTISEMENTS"', '"ADDS"', '"ADVERTISEMENT"', '"ADS/ADVERTISING"', '"THE ADS"', '"ADVERTISEMENTS"', '"ADVERTS"', '"AD FILTERING"', '"POLITICAL ADS"', '"THESE ADS"', '"RECOMMENDATIONS"', '"3 PROPAGANDAS"', '"EXPLORE FEATURE"', '"AD-FREE EXPERIENCE"', '"COMMERCIAL/ADVERTISEMENT"', '"CUSTOMIZED ADS"', '"AUTONOMY"', '"ADS"', '"ADS IN PODCASTS"']

Communities (level, cluster_id) (24): [(0, 'Cluster 1'), (0, 'Cluster 12'), (0, 'Cluster 7'), (0, 'Cluster 10'), (1, 'Cluster 82'), (1, 'Cluster 92'), (2, 'Cluster 168'), (1, 'Cluster 48'), (2, 'Cluster 156'), (1, 'Cluster 122'), (2, 'Cluster 210'), (2, 'Cluster 165'), (0, 'Cluster 32'), (1, 'Cluster 119'), (1, 'Cluster 107'), (2, 'Cluster 192'), (0, 'Clus

## Issues with Advertisements on Spotify

Spotify users, particularly those utilizing the free version of the service, frequently express dissatisfaction with the presence of advertisements. As seen across numerous user reviews on the Appstore and Playstore, ads are often described as intrusive and repetitive, detracting from the overall listening experience. Many users find these ads disruptive during music playback, with specific complaints highlighting instances where ads interrupt relaxation-focused content like meditation sessions ("Spotify Music Ads Ecosystem", "App Store and Spotify Review Ecosystem"). This disruption not only affects casual listening but also impacts users during activities where concentrated listening is desired, like meditation or focus sessions.

Furthermore, some Spotify Premium subscribers report encountering advertisements despite paying for an ad-free experience, which undermines the value proposition of their subscription. This unexpected presence of ad

In [11]:
print(graph_func.query(" how to make pricing better tell suggestions? ", param=QueryParam(mode="hi")))



2025-03-31 01:41:21,507 - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-03-31 01:41:21,761 - Using 20 entites, 23 communities, 167 reasoning path items, 16 text units
2025-03-31 01:41:21,763 - ====== References ======:
Entities (20): ['"PRICE"', '"SUBSCRIPTION PRICES"', '"£5"', '"STUDENT DISCOUNT"', '"PREMIUM PRICE"', '"PLAN"', '"PRODUCTION HOUSES IMPROVEMENT"', '"PREMIUM OPTIONS"', '"UNIVERSITY STUDENT DISCOUNT"', '"NEW HOME MENU"', '"PREMIUM"', '"SUPER PREMIUM"', '"STUDENT PACKAGE"', '"NEW MUSIC SUGGESTIONS"', '"PREMIUM SUBSCRIPTION"', '"PODCAST SUGGESTIONS"', '"€18 SUBSCRIPTION"', '"PREMIUM SERVICE"', '"SPOTIFY PREMIUM"', '"PREMIUM FEATURES"']

Communities (level, cluster_id) (23): [(0, 'Cluster 12'), (1, 'Cluster 122'), (2, 'Cluster 210'), (0, 'Cluster 11'), (0, 'Cluster 10'), (1, 'Cluster 101'), (2, 'Cluster 184'), (0, 'Cluster 4'), (2, 'Cluster 202'), (1, 'Cluster 120'), (0, 'Cluster 1'), (1, 'Cluster 48'), (2, 'Cluster 156'), (0, 'Cluster 9'), (1

To improve pricing strategies for Spotify, a comprehensive understanding of user feedback and market dynamics is crucial. Below are several suggestions derived from user discussions and industry trends that could enhance Spotify's pricing model:

### Tiered Pricing Strategies
One approach to make Spotify pricing more appealing is through tiered pricing strategies that cater to different user needs. For instance, offering a **middle-tier option** between the free version and the current Premium service may attract users who find the Premium price too high but are willing to pay for fewer ads and some extra features. This tier could include limited ad-free listening and a moderate number of skips and downloads. Implementing a **family or group plan** with attractive discounts could also encourage multiple sign-ups within households.

### Promotional Discounts and Trials
A popular approach is incorporating promotional discounts and extended *free trial* periods. User feedback indicates a 

In [12]:
print(graph_func.query("all bugs reported by users ", param=QueryParam(mode="hi")))


2025-03-31 01:43:06,599 - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-03-31 01:43:06,811 - Using 20 entites, 23 communities, 139 reasoning path items, 16 text units
2025-03-31 01:43:06,813 - ====== References ======:
Entities (20): ['"BUG"', '"MUSIC BUG"', '"THE ADS"', '"USER REVIEWS"', '"REVIEWERS"', '"UI/UX"', '"SPANISH USER"', '"THESE ADS"', '"9.0.14 / 9.0.16"', '"DRIVING MODE"', '"GLITCH"', '"SEARCH UI"', '"USER INTERFACE"', '"FREE USERS"', '"HEART FOR LIKED SONGS FEATURE"', '"FRENCH USER"', '"ADVERTISEMENT"', '"ECUADOR"', '"ADDS"', '"ADVERTS"']

Communities (level, cluster_id) (23): [(0, 'Cluster 12'), (1, 'Cluster 122'), (2, 'Cluster 210'), (0, 'Cluster 4'), (0, 'Cluster 1'), (1, 'Cluster 48'), (2, 'Cluster 156'), (1, 'Cluster 108'), (2, 'Cluster 165'), (0, 'Cluster 7'), (1, 'Cluster 80'), (2, 'Cluster 197'), (0, 'Cluster 11'), (1, 'Cluster 83'), (1, 'Cluster 151'), (0, 'Cluster 32'), (1, 'Cluster 82'), (1, 'Cluster 100'), (2, 'Cluster 182'), (0

### Bugs Reported by Users

**Connectivity and Playback Issues:**
Many users report that Spotify has issues with song playback and connectivity, especially with its offline mode. A common problem is songs stopping every 3 seconds, which requires a phone restart to fix the issue. Users have also noted that downloaded music can randomly become unplayable when offline, and multiple attempts are needed to make the album or playlist even appear.

**Smart Shuffle and Playlist Management:**
Smart Shuffle has been a particular point of frustration for users. Some users report that it adds unwanted songs and forces them to listen to tracks that do not align with their preferences. Users also express issues with the app adding songs to playlists automatically without their consent, disrupting their curated lists. Another notable issue is the shuffling feature repeatedly playing the same set of tracks within large playlists, diminishing the experience of variety.

**User Interface and Experience:

In [ ]:

# comment this if the working directory has already been indexed
with open("your .txt file path") as f:
    graph_func.insert(f.read())


print("Perform hi search:")
print(graph_func.query("What are the top themes in this story?", param=QueryParam(mode="hi")))